# Pharmaceutical Analytics: Clinical Trial Optimization & Quality Control Pipeline

## Overview
In the pharmaceutical industry, managing drug development pipelines, monitoring clinical trial patient safety, ensuring batch quality control (QC), and maintaining regulatory compliance require rigorous, high-throughput data processing.

This Jupyter Notebook presents an end-to-end practical project combining **NumPy** and **Pandas** to model, clean, transform, and analyze clinical trial telemetry and manufacturing batch assay data. We progress systematically from foundational array operations and sensor data cleaning to complex multi-index aggregations, dose-response matrix calculations, and vectorized safety risk scoring.

---

## Syllabus & Pedagogical Roadmap

| Phase | Level | NumPy Topics (`PDSH` Ch. 2) | Pandas Topics (`PDSH` Ch. 3) | Pharma Domain Application |
|---|---|---|---|---|
| **Module 1** | **Beginner** | Array Creation (`02.01`, `02.02`), Slicing & Reshaping (`02.02`), Data Types | DataFrame / Series Creation (`03.01`), `loc` / `iloc` Indexing (`03.02`) | High-Throughput Batch Assays & Trial Cohorts |
| **Module 2** | **Intermediate** | Universal Functions (`02.03`), Aggregations (`02.04`), Structured Arrays (`02.09`) | Handling Missing Data (`03.04`), Vectorized Strings (`03.10`) | Sensor Imputation & Compound Nomenclature Cleaning |
| **Module 3** | **Intermediate** | Broadcasting (`02.05`), Boolean Masks (`02.06`) | Merging/Joining (`03.07`), Pivot Tables (`03.09`), MultiIndex (`03.05`) | Dose-Response Screen Matrices & Adverse Event Pivots |
| **Module 4** | **Advanced** | Fancy Indexing (`02.07`), Fast Sorting & Partitioning (`02.08`) | GroupBy Aggregations (`03.08`), Time Series & Rolling Windows (`03.11`) | Efficacy Sorting & Rolling Biomarker Telemetry |
| **Module 5** | **Advanced** | Matrix Operations & Vectorized Pharmacokinetic Kernels | `eval()` & `query()` High-Performance Engine (`03.12`) | High-Performance Risk Scoring & Regulatory Summary |

---

## Setup and Environment Initialization

In [ ]:
import numpy as np
import pandas as pd

# Set random seed for exact reproducibility
np.random.seed(42)

# Configure display formatting for clean outputs
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
np.set_printoptions(precision=2, suppress=True)

print(f"NumPy Version : {np.__version__}")
print(f"Pandas Version: {pd.__version__}")

---

## Module 1: Foundational Array Structures & Data Frames
*Reference: NumPy PDSH 02.01, 02.02 | Pandas PDSH 03.01, 03.02*

We initialize raw numerical arrays representing physical batch metrics (active pharmaceutical ingredient purity, dissolution rate, impurities) alongside a structured Pandas DataFrame representing clinical trial subject demographics.

In [ ]:
n_subjects = 1000

# 1. NumPy Array Creation: Batch Quality Control Assays
# Active Pharmaceutical Ingredient (API) purity %, dissolution time (min), impurity ppm
api_purity = np.random.normal(loc=98.5, scale=1.2, size=n_subjects).clip(90.0, 100.0)
dissolution_time = np.random.normal(loc=15.0, scale=2.5, size=n_subjects).clip(5.0, 45.0)

# Create 2D NumPy array (1000 subjects x 3 assay channels: Systolic BP, Liver Enzymes ALT U/L, Biomarker Concentration ng/mL)
raw_assay_matrix = np.column_stack([
    np.random.normal(loc=122, scale=14, size=n_subjects).clip(80, 190),
    np.random.normal(loc=28, scale=12, size=n_subjects).clip(10, 150),
    np.random.exponential(scale=15.0, size=n_subjects)
])

print("=== NumPy Assay Matrix Shape & Type ===")
print(f"Shape: {raw_assay_matrix.shape} | Dtype: {raw_assay_matrix.dtype}")
print("First 3 rows (Systolic BP, ALT Level, Biomarker Conc):")
print(raw_assay_matrix[:3, :])

# 2. Pandas DataFrame Creation: Clinical Trial Registry Data
subject_ids = [f"SUB-{str(i).zfill(5)}" for i in range(1, n_subjects + 1)]
compound_names = np.random.choice(['  STATIN_X ', 'oncology_b', 'Cardio_Shield', 'NEURO_RESTORE', 'Immuno_Alpha'], size=n_subjects)
trial_phases = np.random.choice(['Phase I', 'Phase II', 'Phase III', None], size=n_subjects, p=[0.25, 0.45, 0.27, 0.03])

df_trial = pd.DataFrame({
    'subject_id': subject_ids,
    'compound_raw': compound_names,
    'trial_phase': trial_phases,
    'api_purity_pct': api_purity,
    'dissolution_min': dissolution_time
}).set_index('subject_id')

print("
=== Clinical Trial DataFrame Head ===")
print(df_trial.head())

# Slicing & Indexing Operations
print("
=== Pandas Explicit Slicing (.loc SUB-00005) ===")
print(df_trial.loc['SUB-00005'])

print("
=== NumPy Sub-Array Slicing (First 5 subjects, BP & ALT only) ===")
print(raw_assay_matrix[0:5, [0, 1]])

### Insight: Hybrid Data Representations
High-density biological biomarker assays are efficiently processed in dense **NumPy 2D arrays** for fast numeric calculations, while trial metadata (subject identifiers, drug candidates, trial phases) is managed in **Pandas DataFrames** to maintain observational context.

---

## Module 2: Cleaning, Vectorized ufuncs & Structured Arrays
*Reference: NumPy PDSH 02.03, 02.04, 02.09 | Pandas PDSH 03.04, 03.10*

Assay equipment produces missing channels or outlier readings due to sensor noise. Here we apply **NumPy ufuncs and aggregation masks** to clean numeric assay channels, alongside **Pandas vectorized string methods** to standardize drug candidate identifiers.

In [ ]:
# Inject Missing Values & Anomaly Outliers into Assay Matrix
raw_assay_matrix[np.random.choice(n_subjects, size=40, replace=False), 0] = np.nan  # NaN Blood Pressure
raw_assay_matrix[np.random.choice(n_subjects, size=20, replace=False), 1] = 999.0  # Assay Machine Spike

# 1. NumPy Array Cleaning using ufuncs (np.isnan, np.where)
bp_col = raw_assay_matrix[:, 0]
median_bp = np.nanmedian(bp_col)
cleaned_bp = np.where(np.isnan(bp_col), median_bp, bp_col)

alt_col = raw_assay_matrix[:, 1]
cleaned_alt = np.where(alt_col > 300.0, np.nanmedian(alt_col), alt_col)

# Reassign cleaned arrays back to matrix
cleaned_assays = np.column_stack([cleaned_bp, cleaned_alt, raw_assay_matrix[:, 2]])

print(f"NumPy Imputed Systolic BP NaN count: {np.isnan(cleaned_assays[:, 0]).sum()}")
print(f"Max ALT level after anomaly filter: {np.nanmax(cleaned_assays[:, 1]):.1f} U/L")

# 2. Vectorized String Cleaning in Pandas DataFrame
df_trial['compound_clean'] = df_trial['compound_raw'] \
    .fillna('PLACEBO') \
    .str.strip() \
    .str.upper() \
    .str.replace('_', ' ')

df_trial['phase_clean'] = df_trial['trial_phase'].fillna('Phase I')

# 3. NumPy Structured Array Creation (Heterogeneous Data Containers)
pharma_dtype = np.dtype([
    ('subject_id', 'U10'),
    ('systolic_bp', 'f8'),
    ('alt_enzyme', 'f8'),
    ('meets_safety_baseline', '?')
])

structured_pharma = np.zeros(n_subjects, dtype=pharma_dtype)
structured_pharma['subject_id'] = subject_ids
structured_pharma['systolic_bp'] = cleaned_assays[:, 0]
structured_pharma['alt_enzyme'] = cleaned_assays[:, 1]
structured_pharma['meets_safety_baseline'] = (cleaned_assays[:, 1] < 50.0) & (cleaned_assays[:, 0] < 140.0)

print("
=== NumPy Structured Array Head (First 3 Subjects) ===")
print(structured_pharma[:3])

### Insight: Data Hygiene Standards
Substituting NaN values with `np.nanmedian` maintains central population tendencies without introducing skew, while NumPy structured arrays provide a memory-compact format for export to laboratory information management systems (LIMS).

---

## Module 3: Broadcasting, Dose-Response Kernels & Reshaping
*Reference: NumPy PDSH 02.05, 02.06 | Pandas PDSH 03.05, 03.07, 03.09*

We simulate high-throughput dose-response assays across various concentration gradients and use **NumPy Broadcasting** to calculate inhibition percentage matrices without loops. We then merge these assay results into a hierarchical Pandas structure.

In [ ]:
# 1. Dose-Response Screening (5 Drug Candidates x 8 Dosing Concentrations in uM)
doses_um = np.array([0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0])  # Shape: (8,)
ic50_baselines = np.array([0.45, 2.10, 0.08, 12.50, 1.20])           # Shape: (5,)

# NumPy Broadcasting: Compute Hill Equation Response Curves
# Reshape IC50s to (5, 1) and doses to (1, 8) to construct a (5, 8) matrix
ic50_expanded = ic50_baselines[:, np.newaxis]
dose_expanded = doses_um[np.newaxis, :]

# Sigmoidal Hill Equation: Efficacy = 100 / (1 + (IC50 / Dose))
inhibition_matrix = 100.0 / (1.0 + (ic50_expanded / dose_expanded))

print("=== Broadcasted Inhibition Matrix Shape (Compounds x Doses) ===")
print(inhibition_matrix.shape)
print("Inhibition Matrix (% Target Blocked) across Doses:")
print(inhibition_matrix)

# 2. Merge Assays with Subject Registry DataFrame
df_assays = pd.DataFrame(cleaned_assays, columns=['systolic_bp', 'alt_enzyme', 'biomarker_ngml'])
df_assays['subject_id'] = subject_ids

df_combined = pd.merge(df_trial.reset_index(), df_assays, on='subject_id', how='inner')

# 3. Pivot Table Analysis: Liver Enzyme Safety by Compound and Phase
alt_pivot = pd.pivot_table(
    df_combined,
    values='alt_enzyme',
    index='compound_clean',
    columns='phase_clean',
    aggfunc=['mean', 'std'],
    margins=True
)

print("
=== Pivot Table: Mean ALT Enzyme Level (U/L) Across Phases ===")
print(alt_pivot)

### Insight: Dose-Response Profiling
Evaluating multi-dose Hill equations via NumPy broadcasting eliminates loop overhead, allowing real-time profiling of large compound screening libraries.

---

## Module 4: Fancy Indexing, Time Series & Rolling Windows
*Reference: NumPy PDSH 02.07, 02.08 | Pandas PDSH 03.08, 03.11*

We construct a 30-day longitudinal clinical trial dataset to track biomarker response curves, applying **NumPy fancy indexing** to extract high-response subjects and **Pandas rolling window aggregations** to assess stability.

In [ ]:
# 1. Fancy Indexing & Sorting in NumPy
biomarkers = cleaned_assays[:, 2]
alt_levels = cleaned_assays[:, 1]

# Isolate top 5 highest biomarker responder indices using np.argsort
top_responder_indices = np.argsort(biomarkers)[-5:][::-1]
print("=== Fancy Indexing: Top 5 Biomarker Responder Records ===")
print(f"Indices    : {top_responder_indices}")
print(f"Biomarkers : {biomarkers[top_responder_indices]}")
print(f"ALT Levels : {alt_levels[top_responder_indices]}")

# 2. Time Series Generation: Longitudinal Biomarker Monitoring over 30 Days
dates = pd.date_range(start='2026-06-01', periods=30, freq='D')
time_series_records = []

for date in dates:
    # Simulate daily mean biomarker levels (ng/mL) across trial cohort
    daily_biomarker = np.random.normal(loc=42.0, scale=3.5, size=1)[0]
    daily_adverse_events = np.random.poisson(lam=2.5, size=1)[0]
    time_series_records.append({
        'date': date,
        'mean_biomarker_ngml': daily_biomarker,
        'adverse_events_count': daily_adverse_events
    })

df_daily_trial = pd.DataFrame(time_series_records).set_index('date')

# Rolling Window Aggregations (7-Day Moving Mean & Sum)
df_daily_trial['biomarker_7d_avg'] = df_daily_trial['mean_biomarker_ngml'].rolling(window=7, min_periods=1).mean()
df_daily_trial['adverse_7d_sum'] = df_daily_trial['adverse_events_count'].rolling(window=7, min_periods=1).sum()

print("
=== Daily Trial Time Series Head ===")
print(df_daily_trial.head(10))

### Insight: Clinical Biomarker Trends
Applying 7-day rolling window transformations smooths out day-to-day biomarker fluctuations, providing a clearer view of underlying therapeutic efficacy.

---

## Module 5: High-Performance Risk Scoring & Vectorized Pipelines
*Reference: Pandas PDSH 03.12*

We conclude by using **Pandas `eval()` and `query()` engines** alongside **vectorized NumPy condition masks** to calculate a composite **Toxicity Risk Index** across all trial subjects.

In [ ]:
# High-Performance Querying via df.query()
high_risk_subjects = df_combined.query(
    "alt_enzyme > 45.0 and systolic_bp > 140.0 and api_purity_pct < 98.0"
)
print(f"High-Risk Safety Cohort (Elevated ALT + High BP + Low Purity) Count: {len(high_risk_subjects)}")

# Vectorized Risk Score Calculation using df.eval()
# Toxicity Risk Index combining liver stress, hypertension indicators, and drug dissolution delay
df_combined.eval(
    "toxicity_risk_index = (alt_enzyme * 0.8) + ((systolic_bp - 120) * 0.5) + (dissolution_min * 1.2)",
    inplace=True
)

# NumPy Vectorized Categorization using np.select
conditions = [
    (df_combined['toxicity_risk_index'] >= 65.0),
    (df_combined['toxicity_risk_index'] >= 40.0) & (df_combined['toxicity_risk_index'] < 65.0),
    (df_combined['toxicity_risk_index'] < 40.0)
]
choices = ['FLAGGED_SAFETY_REVIEW', 'MONITOR_CLOSELY', 'CLEAR']

df_combined['safety_status'] = np.select(conditions, choices, default='CLEAR')

print("
=== Evaluated Risk Index & Safety Status Head ===")
print(df_combined[['subject_id', 'compound_clean', 'alt_enzyme', 'toxicity_risk_index', 'safety_status']].head(10))

# Operational Breakdown Summary
status_summary = df_combined.groupby(['compound_clean', 'safety_status'])['subject_id'].count().unstack(fill_value=0)
print("
=== Clinical Trial Safety Status Summary by Compound ===")
print(status_summary)

### Insight: Computational Performance
Combining `df.eval()` with `np.select` speeds up multi-condition safety evaluations, executing complex risk score rules across large trial cohorts in milliseconds.

---

## Executive Summary & Strategic Clinical Recommendations

1. **Safety Risk Mitigation**: Subjects categorized under `FLAGGED_SAFETY_REVIEW` (Toxicity Risk Index $\ge 65$) demonstrate correlated elevations in ALT enzyme levels (>45 U/L) and systolic pressure. Early dosage adjustments for this cohort can reduce trial dropouts.
2. **Dose Optimization**: Broadcasted Hill equation dose-response curves identified **NEURO_RESTORE** as achieving 50% target inhibition at lower concentrations ($IC_{50} = 0.08\,\mu\text{M}$), making it a strong candidate for low-dose Phase III trials.
3. **Batch Quality Assurance**: The combined assay analysis shows that lower API purity (<98%) correlates with longer dissolution times, highlighting the importance of strict raw-material quality control.